In [9]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf

In [10]:
def generate_event_tables_with_trace(csv_dir):
    """
    For each CSV in the directory, creates a simplified table with:
    - event number
    - trace number
    - event magnitude (pA)

    Saves the table as '<original_filename>_event_table_with_trace.csv'
    in the same directory.
    """
    csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)

            if "trace" not in df.columns or "peak amp (pA)" not in df.columns:
                print(f"Skipping {csv_file} — missing required columns.")
                continue

            event_table = pd.DataFrame({
                "event number": range(len(df)),
                "trace": df["trace"],
                "event magnitude (pA)": df["peak amp (pA)"]
            })

            base_name = os.path.splitext(os.path.basename(csv_file))[0]
            output_path = os.path.join(csv_dir, f"{base_name}_event_table_with_trace.csv")
            event_table.to_csv(output_path, index=False)
            print(f"Saved: {output_path}")

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

In [13]:
generate_event_tables_with_trace("/Users/jayashri/Desktop/MCcsv")

Skipping /Users/jayashri/Desktop/MCcsv/25609003Events_event_table_with_magnitude.csv — missing required columns.
Saved: /Users/jayashri/Desktop/MCcsv/25609003Events_event_table_with_trace.csv


/var/folders/bz/z94gnqs93496sv93fzsm61880000gn/T/ipykernel_47253/3952745072.py:15: DtypeWarning: Columns (16,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


In [16]:
def extract_sweeps_by_event_numbers(abf_path, event_numbers):
    abf = pyabf.ABF(abf_path)
    waveforms = []

    for sweep in event_numbers:
        if sweep < abf.sweepCount:
            abf.setSweep(sweep)
            waveforms.append(abf.sweepY.copy())
        else:
            print(f"Skipping sweep {sweep} (not in {os.path.basename(abf_path)})")

    return np.array(waveforms), abf.sweepX

def average_waveforms_from_trace_ranges(event_table, abf_path, trace_windows, min_magnitude_pA=10):
    averaged_waveforms = []
    sweep_groups = []

    # Only include events with magnitude ≥ threshold
    filtered = event_table[np.abs(event_table["event magnitude (pA)"]) >= min_magnitude_pA]

    for start, end in trace_windows:
        matching = filtered[
            (filtered["trace"] >= start) &
            (filtered["trace"] <= end)
        ]["event number"].astype(int).to_list()
        sweep_groups.append(matching)

    for group in sweep_groups:
        waveforms, time_axis = extract_sweeps_by_event_numbers(abf_path, group)
        if len(waveforms) > 0:
            mean_waveform = np.mean(waveforms, axis=0)
        else:
            mean_waveform = None
        averaged_waveforms.append(mean_waveform)

    return time_axis, averaged_waveforms

def plot_average_waveforms(time_axis, waveforms, labels, colors, title, save_path):
    plt.figure(figsize=(10, 6))
    for waveform, label, color in zip(waveforms, labels, colors):
        if waveform is not None:
            plt.plot(time_axis, waveform, label=label, color=color, lw=2)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (pA)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved: {save_path}")

def process_large_events_by_trace(csv_dir, abf_dir, output_dir, min_magnitude_pA=10):
    """
    Processes all simplified event tables in csv_dir, and for each:
    - filters out events with |peak amp| < min_magnitude_pA
    - averages waveforms within standard trace number windows
    - saves overlaid waveform plot to output_dir
    """
    os.makedirs(output_dir, exist_ok=True)

    trace_windows = [
        (1, 10),
        (12, 22),
        (24, 34)
    ]
    labels = [
        "Trace 1–10",
        "Trace 12–22",
        "Trace 24–34"
    ]
    colors = ["black", "red", "blue"]

    simplified_csvs = glob.glob(os.path.join(csv_dir, "*_event_table_with_trace.csv"))

    for csv_file in simplified_csvs:
        try:
            df = pd.read_csv(csv_file)
            if not {"event number", "trace", "event magnitude (pA)"}.issubset(df.columns):
                print(f"Skipping {csv_file} — malformed format.")
                continue

            base_name = os.path.basename(csv_file).replace("_event_table_with_trace.csv", "")
            abf_path = os.path.join(abf_dir, f"{base_name}.abf")

            if not os.path.exists(abf_path):
                print(f"No ABF file found for {base_name}")
                continue

            time_axis, averaged_waveforms = average_waveforms_from_trace_ranges(
                df, abf_path, trace_windows, min_magnitude_pA=min_magnitude_pA
            )

            save_path = os.path.join(output_dir, f"{base_name}_avg_waveforms_by_trace.png")
            title = f"Avg Waveforms by Trace (≥{min_magnitude_pA} pA): {base_name}"
            plot_average_waveforms(time_axis, averaged_waveforms, labels, colors, title, save_path)

        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

In [18]:
process_large_events_by_trace(
    csv_dir="/Users/jayashri/Desktop/MCcsv",
    abf_dir="/Users/jayashri/Desktop/MCEvents",
    output_dir="/Users/jayashri/Desktop/MCEvents"
)

Saved: /Users/jayashri/Desktop/MCEvents/25609003Events_avg_waveforms_by_trace.png
